# vabert usage

This notebook shows how to run the `vabert` visual acuity (VA) pipeline on clinic letter text, in two ways:

1. End to end: letter text in, a parsed table of VA readings out.
2. The same table next to the raw model predictions, with confidence scores and the evidence the rule check found.

It then works through four groups of examples: letters that state everything, letters that leave out the eye or the method, readings in different notations, and readings the pipeline sends to review.

Run it from the release folder, or from anywhere once the package is installed with `pip install -e <release folder>`. Everything it needs is in the folder; no network access is used.

## How the pipeline reports a reading

Each detected reading has three attributes:

| Field | Values |
|---|---|
| `lat` (laterality) | `Right`, `Left`, `Both`, `Unk` |
| `method` | `Aided`, `Unaided`, `AidedPH`, `UnaidedPH`, `UnkPH`, `Unk` |
| `notation` | `snellen_m`, `snellen_ft`, `etdrs`, `logmar`, `decimal`, `thomson`, `qualitative`, `Unk` |

For each field the final answer is a value, `Unk` or `Review`:

- a value is committed only when the model is confident (at or above the field's threshold in `thresholds.json`) and the rule-based evidence check reads the same value from the text;
- `Unk` means both agree the letter doesn't state it. This is a correct, final answer, not a failure;
- `Review` means the system isn't sure (low confidence, or the model and the evidence check disagree), so a person should look.

The value itself also goes to review if it doesn't fit its notation (for example `6/52` read as Snellen). The letter gets a status: `complete`, `complete_with_unk`, `needs_review` or `no_va`.

In [1]:
import os, sys, warnings
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')    # quieter start-up; transformers doesn't need TensorFlow here
os.environ.setdefault('USE_TF', '0')
warnings.filterwarnings('ignore')
import pandas as pd

from vabert import VABERTPipeline           # works from this folder, or anywhere after `pip install -e .`

pipe = VABERTPipeline()                   # loads model/ and thresholds.json; uses a GPU if one is visible
pd.set_option('display.max_colwidth', 60)
pipe.tau

{'lat': 0.943, 'method': 0.985, 'notation': 0.644}

## 1. End to end: text in, table out

`pipe.run(text)` returns the letter status and one dict per reading. `parse()` below turns that into a table with the final answers.

In [2]:
FINAL_COLS = ['va_value', 'lat', 'method', 'notation', 'label', 'start', 'end']

def parse(text):
    """Letter text -> (letter status, table of VA readings with final answers)."""
    result = pipe.run(text)
    table = pd.DataFrame(result['readings'], columns=None)
    if table.empty:
        return result['status'], pd.DataFrame(columns=FINAL_COLS)
    return result['status'], table[FINAL_COLS]

letter = "Visual acuity with glasses: right eye 6/9, left eye 6/12."
status, table = parse(letter)
print('letter status:', status)
table

letter status: complete


,va_value,lat,method,notation,label,start,end
0,6/9,Right,Aided,snellen_m,VA-Right-Aided,38,41
1,6/12,Left,Aided,snellen_m,VA-Left-Aided,52,56


Several letters at once: `parse_many()` returns one table with a letter id per row, and letters without VA keep a row so nothing silently disappears. The command line does the same for a CSV file: `vabert --corpus letters.csv --text-column text --id-column id --out results.csv`.

In [3]:
def parse_many(letters):
    """{letter_id: text} -> one table, one row per reading (or one row per letter with no VA)."""
    rows = []
    for lid, text in letters.items():
        result = pipe.run(text)
        if not result['readings']:
            rows.append({'letter_id': lid, 'letter_status': result['status']})
        for r in result['readings']:
            rows.append({'letter_id': lid, 'letter_status': result['status'], **{k: r[k] for k in FINAL_COLS}})
    return pd.DataFrame(rows).astype({'start': 'Int64', 'end': 'Int64'})

parse_many({
    'A': "Visual acuity with glasses: right eye 6/9, left eye 6/12.",
    'B': "Her unaided visual acuity was 6/9 today.",
    'C': "Intraocular pressure was 16 mmHg in both eyes. Review in 4 months.",
})

,letter_id,letter_status,va_value,lat,method,notation,label,start,end
0,A,complete,6/9,Right,Aided,snellen_m,VA-Right-Aided,38,41
1,A,complete,6/12,Left,Aided,snellen_m,VA-Left-Aided,52,56
2,B,complete_with_unk,6/9,Unk,Unaided,snellen_m,VA-Unk-Unaided,30,33
3,C,no_va,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>


## 2. Table plus raw predictions and confidence

For each field the pipeline also keeps the model's own answer (`*_model`), its confidence (`*_confidence`), the state (`committed`, `unk` or `review`), the reason, and the text the evidence check used (`*_evidence`). `explain()` shows them per field.

The raw model output, before the plausibility filter and the commit gate, comes from `pipe.model.predict(text)`: per span the predicted value of each field, its confidence, and the full probability distribution over that field's values (`*_dist`).

In [4]:
FIELDS = ['lat', 'method', 'notation']

def explain(text):
    """Letter text -> (status, long table: one row per reading and field with final value, model answer, confidence, state, reason, evidence)."""
    result = pipe.run(text)
    rows = []
    for r in result['readings']:
        rows.append({'va_value': r['va_value'], 'field': 'value', 'final': r['va_value'], 'model': r['va_value'],
                     'confidence': r['span_confidence'], 'state': r['value_state'], 'reason': r['value_reason'], 'evidence': None})
        for f in FIELDS:
            rows.append({'va_value': r['va_value'], 'field': f, 'final': r[f], 'model': r[f + '_model'],
                         'confidence': r[f + '_confidence'], 'state': r[f + '_state'], 'reason': r[f + '_reason'],
                         'evidence': r[f + '_evidence']})
    return result['status'], pd.DataFrame(rows)

def raw(text):
    """Raw model spans (no filter, no gate) with per-field prediction and confidence."""
    rows = []
    for x in pipe.model.predict(text):
        rows.append({'va_value': x['val'], 'start': x['s'], 'end': x['e'], 'span_conf': round(x['span_conf'], 4),
                     **{f: x[f] for f in FIELDS}, **{f + '_conf': round(x[f + '_conf'], 4) for f in FIELDS}})
    return pd.DataFrame(rows)

letter = "Visual acuity with glasses: right eye 6/9, left eye 6/12."
status, detail = explain(letter)
print('letter status:', status)
detail

letter status: complete


,va_value,field,final,model,confidence,state,reason,evidence
0,6/9,value,6/9,6/9,0.9983,committed,None,None
1,6/9,lat,Right,Right,0.9993,committed,model_and_evidence_agree,right
2,6/9,method,Aided,Aided,0.9991,committed,model_and_evidence_agree,with glasses
3,6/9,notation,snellen_m,snellen_m,0.9988,committed,model_and_evidence_agree,None
4,6/12,value,6/12,6/12,0.9984,committed,None,None
5,6/12,lat,Left,Left,0.9990,committed,model_and_evidence_agree,left
6,6/12,method,Aided,Aided,0.9990,committed,model_and_evidence_agree,with glasses
7,6/12,notation,snellen_m,snellen_m,0.9991,committed,model_and_evidence_agree,None


In [5]:
raw(letter)

,va_value,start,end,span_conf,lat,method,notation,lat_conf,method_conf,notation_conf
0,6/9,38,41,0.9983,Right,Aided,snellen_m,0.9993,0.9991,0.9988
1,6/12,52,56,0.9984,Left,Aided,snellen_m,0.9990,0.9990,0.9991


In [6]:
# full probability distribution over each field's values, for the first span
from vabert.inference import M   # the model definition (vabert/model_def.py); M.HEADS lists each head's values in order
first = pipe.model.predict(letter)[0]
pd.DataFrame([{'field': f, **dict(zip(M.HEADS[f], first[f + '_dist']))} for f in FIELDS]).set_index('field').fillna('')

,Right,Left,Both,Unk,Aided,Unaided,AidedPH,UnaidedPH,UnkPH,snellen_m,snellen_ft,qualitative,etdrs,logmar,decimal,thomson
field,,,,,,,,,,,,,,,,
lat,0.9993,0.0003,0.0002,0.0002,,,,,,,,,,,,
method,,,,0.0001,0.9991,0.0001,0.0005,0.0001,0.0001,,,,,,,
notation,,,,0.0002,,,,,,0.9988,0.0001,0.0001,0.0002,0.0001,0.0002,0.0002


A convenience view used for the examples below: both tables for one letter. In the second table the `value` row is the reading itself (state `review` if it fails its notation's grammar).

In [7]:
def show(text):
    print(text, '\n')
    status, table = parse(text)
    print('letter status:', status)
    display(table)
    _, detail = explain(text)
    if not detail.empty:
        display(detail[['va_value', 'field', 'final', 'model', 'confidence', 'state', 'reason', 'evidence']])

## Examples

### (1) Ideal case: eye and method stated

Every field is committed and the letter is `complete`. A narrative sentence first, then the vertical table layout used by the electronic patient record (eye from the column header, method from the row label; `NIPH`, "no improvement with pinhole", is not a reading).

In [8]:
show("Visual acuity with glasses: right eye 6/9, left eye 6/12.")

Visual acuity with glasses: right eye 6/9, left eye 6/12. 

letter status: complete


,va_value,lat,method,notation,label,start,end
0,6/9,Right,Aided,snellen_m,VA-Right-Aided,38,41
1,6/12,Left,Aided,snellen_m,VA-Left-Aided,52,56


,va_value,field,final,model,confidence,state,reason,evidence
0,6/9,value,6/9,6/9,0.9983,committed,None,None
1,6/9,lat,Right,Right,0.9993,committed,model_and_evidence_agree,right
2,6/9,method,Aided,Aided,0.9991,committed,model_and_evidence_agree,with glasses
3,6/9,notation,snellen_m,snellen_m,0.9988,committed,model_and_evidence_agree,None
4,6/12,value,6/12,6/12,0.9984,committed,None,None
5,6/12,lat,Left,Left,0.9990,committed,model_and_evidence_agree,left
6,6/12,method,Aided,Aided,0.9990,committed,model_and_evidence_agree,with glasses
7,6/12,notation,snellen_m,snellen_m,0.9991,committed,model_and_evidence_agree,None


In [9]:
show("Visual Acuity\nRight Eye\nLeft Eye\n\nUnaided\n6/18\n6/12\n\nPinhole\n6/9\nNIPH\n")

Visual Acuity
Right Eye
Left Eye

Unaided
6/18
6/12

Pinhole
6/9
NIPH
 

letter status: complete


,va_value,lat,method,notation,label,start,end
0,6/18,Right,Unaided,snellen_m,VA-Right-Unaided,42,46
1,6/12,Left,Unaided,snellen_m,VA-Left-Unaided,47,51
2,6/9,Right,UnkPH,snellen_m,VA-Right-UnkPH,61,64


,va_value,field,final,model,confidence,state,reason,evidence
0,6/18,value,6/18,6/18,0.9975,committed,None,None
1,6/18,lat,Right,Right,0.9990,committed,model_and_evidence_agree,Right
2,6/18,method,Unaided,Unaided,0.9992,committed,model_and_evidence_agree,Unaided
3,6/18,notation,snellen_m,snellen_m,0.9991,committed,model_and_evidence_agree,None
4,6/12,value,6/12,6/12,0.9982,committed,None,None
5,6/12,lat,Left,Left,0.9994,committed,model_and_evidence_agree,Left
6,6/12,method,Unaided,Unaided,0.9990,committed,model_and_evidence_agree,Unaided
7,6/12,notation,snellen_m,snellen_m,0.9991,committed,model_and_evidence_agree,None
8,6/9,value,6/9,6/9,0.9958,committed,None,None
9,6/9,lat,Right,Right,0.9996,committed,model_and_evidence_agree,Right


### (2) Laterality or method missing

When the letter doesn't say which eye, or how the acuity was measured, the field is `Unk` with reason `not_stated`. The pipeline doesn't guess (no "first value is the right eye" convention), and the letter is `complete_with_unk`, a final result that needs no review.

In [10]:
show("Her unaided visual acuity was 6/9 today.")          # eye not stated

Her unaided visual acuity was 6/9 today. 

letter status: complete_with_unk


,va_value,lat,method,notation,label,start,end
0,6/9,Unk,Unaided,snellen_m,VA-Unk-Unaided,30,33


,va_value,field,final,model,confidence,state,reason,evidence
0,6/9,value,6/9,6/9,0.9971,committed,None,None
1,6/9,lat,Unk,Unk,0.9959,unk,not_stated,None
2,6/9,method,Unaided,Unaided,0.9991,committed,model_and_evidence_agree,unaided
3,6/9,notation,snellen_m,snellen_m,0.9992,committed,model_and_evidence_agree,None


In [11]:
show("Visual acuity: right eye 6/12, left eye 6/6.")      # method not stated

Visual acuity: right eye 6/12, left eye 6/6. 

letter status: complete_with_unk


,va_value,lat,method,notation,label,start,end
0,6/12,Right,Unk,snellen_m,VA-Right-Unk,25,29
1,6/6,Left,Unk,snellen_m,VA-Left-Unk,40,43


,va_value,field,final,model,confidence,state,reason,evidence
0,6/12,value,6/12,6/12,0.9988,committed,None,None
1,6/12,lat,Right,Right,0.9994,committed,model_and_evidence_agree,right
2,6/12,method,Unk,Unk,0.9990,unk,not_stated,None
3,6/12,notation,snellen_m,snellen_m,0.9993,committed,model_and_evidence_agree,None
4,6/6,value,6/6,6/6,0.9987,committed,None,None
5,6/6,lat,Left,Left,0.9992,committed,model_and_evidence_agree,left
6,6/6,method,Unk,Unk,0.9986,unk,not_stated,None
7,6/6,notation,snellen_m,snellen_m,0.9996,committed,model_and_evidence_agree,None


In [12]:
show("Vision was 6/12 at the last visit.")                # neither stated

Vision was 6/12 at the last visit. 

letter status: complete_with_unk


,va_value,lat,method,notation,label,start,end
0,6/12,Unk,Unk,snellen_m,VA-Unk-Unk,11,15


,va_value,field,final,model,confidence,state,reason,evidence
0,6/12,value,6/12,6/12,0.9973,committed,None,None
1,6/12,lat,Unk,Unk,0.9978,unk,not_stated,None
2,6/12,method,Unk,Unk,0.9994,unk,not_stated,None
3,6/12,notation,snellen_m,snellen_m,0.9994,committed,model_and_evidence_agree,None


### (3) Different notations

The `notation` field identifies how the value is written. Bare numbers in a VA context are ETDRS letter scores; qualitative values (counting fingers, hand movements, perception of light) are spans of words.

In [13]:
show("Visual acuity: 73 on the right and 68 on the left.")                                           # ETDRS letters

Visual acuity: 73 on the right and 68 on the left. 

letter status: complete_with_unk


,va_value,lat,method,notation,label,start,end
0,73,Right,Unk,etdrs,VA-Right-Unk,15,17
1,68,Left,Unk,etdrs,VA-Left-Unk,35,37


,va_value,field,final,model,confidence,state,reason,evidence
0,73,value,73,73,0.9978,committed,None,None
1,73,lat,Right,Right,0.9977,committed,model_and_evidence_agree,right
2,73,method,Unk,Unk,0.9979,unk,not_stated,None
3,73,notation,etdrs,etdrs,0.9980,committed,model_and_evidence_agree,None
4,68,value,68,68,0.9985,committed,None,None
5,68,lat,Left,Left,0.9978,committed,model_and_evidence_agree,left
6,68,method,Unk,Unk,0.9980,unk,not_stated,None
7,68,notation,etdrs,etdrs,0.9972,committed,model_and_evidence_agree,None


In [14]:
show("Visual acuity (unaided) was 0.30 logMAR in the right eye and 0.10 logMAR in the left eye.")    # logMAR

Visual acuity (unaided) was 0.30 logMAR in the right eye and 0.10 logMAR in the left eye. 

letter status: complete


,va_value,lat,method,notation,label,start,end
0,0.30,Right,Unaided,logmar,VA-Right-Unaided,28,32
1,0.10,Left,Unaided,logmar,VA-Left-Unaided,61,65


,va_value,field,final,model,confidence,state,reason,evidence
0,0.30,value,0.30,0.30,0.9940,committed,None,None
1,0.30,lat,Right,Right,0.9979,committed,model_and_evidence_agree,right eye
2,0.30,method,Unaided,Unaided,0.9975,committed,model_and_evidence_agree,unaided
3,0.30,notation,logmar,logmar,0.9976,committed,model_and_evidence_agree,logMAR
4,0.10,value,0.10,0.10,0.9932,committed,None,None
5,0.10,lat,Left,Left,0.9983,committed,model_and_evidence_agree,left eye
6,0.10,method,Unaided,Unaided,0.9966,committed,model_and_evidence_agree,unaided
7,0.10,notation,logmar,logmar,0.9973,committed,model_and_evidence_agree,logMAR


In [15]:
show("Unaided visual acuity was 20/40 in the right eye and 20/25 in the left eye.")                 # Snellen in feet

Unaided visual acuity was 20/40 in the right eye and 20/25 in the left eye. 

letter status: complete


,va_value,lat,method,notation,label,start,end
0,20/40,Right,Unaided,snellen_ft,VA-Right-Unaided,26,31
1,20/25,Left,Unaided,snellen_ft,VA-Left-Unaided,53,58


,va_value,field,final,model,confidence,state,reason,evidence
0,20/40,value,20/40,20/40,0.9942,committed,None,None
1,20/40,lat,Right,Right,0.9967,committed,model_and_evidence_agree,right
2,20/40,method,Unaided,Unaided,0.9989,committed,model_and_evidence_agree,Unaided
3,20/40,notation,snellen_ft,snellen_ft,0.9974,committed,model_and_evidence_agree,None
4,20/25,value,20/25,20/25,0.9935,committed,None,None
5,20/25,lat,Left,Left,0.9964,committed,model_and_evidence_agree,left
6,20/25,method,Unaided,Unaided,0.9989,committed,model_and_evidence_agree,Unaided
7,20/25,notation,snellen_ft,snellen_ft,0.9981,committed,model_and_evidence_agree,None


In [16]:
show("Visual acuity with glasses: right eye 0.8 decimal, left eye 0.5 decimal.")                    # decimal

Visual acuity with glasses: right eye 0.8 decimal, left eye 0.5 decimal. 



letter status: complete


,va_value,lat,method,notation,label,start,end
0,0.8,Right,Aided,decimal,VA-Right-Aided,38,41
1,0.5,Left,Aided,decimal,VA-Left-Aided,60,63


,va_value,field,final,model,confidence,state,reason,evidence
0,0.8,value,0.8,0.8,0.9946,committed,None,None
1,0.8,lat,Right,Right,0.9999,committed,model_and_evidence_agree,right
2,0.8,method,Aided,Aided,0.9976,committed,model_and_evidence_agree,with glasses
3,0.8,notation,decimal,decimal,0.8428,committed,model_and_evidence_agree,decimal
4,0.5,value,0.5,0.5,0.9959,committed,None,None
5,0.5,lat,Left,Left,0.9994,committed,model_and_evidence_agree,left
6,0.5,method,Aided,Aided,0.9985,committed,model_and_evidence_agree,with glasses
7,0.5,notation,decimal,decimal,0.9337,committed,model_and_evidence_agree,decimal


In [17]:
show("Visual acuity was hand movements in the right eye and perception of light in the left eye.")  # qualitative

Visual acuity was hand movements in the right eye and perception of light in the left eye. 



letter status: complete_with_unk


,va_value,lat,method,notation,label,start,end
0,hand movements,Right,Unk,qualitative,VA-Right-Unk,18,32
1,perception of light,Left,Unk,qualitative,VA-Left-Unk,54,73


,va_value,field,final,model,confidence,state,reason,evidence
0,hand movements,value,hand movements,hand movements,0.9968,committed,None,None
1,hand movements,lat,Right,Right,0.9998,committed,model_and_evidence_agree,right
2,hand movements,method,Unk,Unk,0.9961,unk,not_stated,None
3,hand movements,notation,qualitative,qualitative,0.9967,committed,model_and_evidence_agree,None
4,perception of light,value,perception of light,perception of light,0.9981,committed,None,None
5,perception of light,lat,Left,Left,0.9993,committed,model_and_evidence_agree,left
6,perception of light,method,Unk,Unk,0.9979,unk,not_stated,None
7,perception of light,notation,qualitative,qualitative,0.9982,committed,model_and_evidence_agree,None


### (4) Readings that need review

`Review` appears when the model isn't confident enough or when the model and the evidence check disagree. Compare the `model` and `confidence` columns with `reason`: the model's own answer is still reported, so a reviewer can accept it in one step.

In [18]:
# pinhole stated in a separate clause: the model's method confidence is below the threshold
show("Unaided visual acuity was 6/9 in the right eye and 6/12 in the left eye. With pinhole the right eye improved to 6/6.")

Unaided visual acuity was 6/9 in the right eye and 6/12 in the left eye. With pinhole the right eye improved to 6/6. 

letter status: needs_review


,va_value,lat,method,notation,label,start,end
0,6/9,Right,Unaided,snellen_m,VA-Right-Unaided,26,29
1,6/12,Left,Unaided,snellen_m,VA-Left-Unaided,51,55
2,6/6,Right,Review,snellen_m,VA-Right-Review,112,115


,va_value,field,final,model,confidence,state,reason,evidence
0,6/9,value,6/9,6/9,0.9977,committed,None,None
1,6/9,lat,Right,Right,0.9994,committed,model_and_evidence_agree,right
2,6/9,method,Unaided,Unaided,0.9988,committed,model_and_evidence_agree,Unaided
3,6/9,notation,snellen_m,snellen_m,0.9990,committed,model_and_evidence_agree,None
4,6/12,value,6/12,6/12,0.9979,committed,None,None
5,6/12,lat,Left,Left,0.9991,committed,model_and_evidence_agree,left
6,6/12,method,Unaided,Unaided,0.9992,committed,model_and_evidence_agree,Unaided
7,6/12,notation,snellen_m,snellen_m,0.9990,committed,model_and_evidence_agree,None
8,6/6,value,6/6,6/6,0.9965,committed,None,None
9,6/6,lat,Right,Right,0.9996,committed,model_and_evidence_agree,right eye


In [19]:
# "VRE" / "VLE": the model reads the eyes correctly, but its confidence (about 0.91 and 0.83) is below the
# laterality threshold and the evidence check doesn't recognise these abbreviations, so the eye goes to review
show("VRE: 6/5, VLE: 6/5")

VRE: 6/5, VLE: 6/5 

letter status: needs_review


,va_value,lat,method,notation,label,start,end
0,6/5,Review,Unk,snellen_m,VA-Review-Unk,5,8
1,6/5,Review,Unk,snellen_m,VA-Review-Unk,15,18


,va_value,field,final,model,confidence,state,reason,evidence
0,6/5,value,6/5,6/5,0.9990,committed,None,None
1,6/5,lat,Review,Right,0.9089,review,low_confidence,None
2,6/5,method,Unk,Unk,0.9998,unk,not_stated,None
3,6/5,notation,snellen_m,snellen_m,0.9992,committed,model_and_evidence_agree,None
4,6/5,value,6/5,6/5,0.9985,committed,None,None
5,6/5,lat,Review,Left,0.8312,review,low_confidence,None
6,6/5,method,Unk,Unk,0.9998,unk,not_stated,None
7,6/5,notation,snellen_m,snellen_m,0.9993,committed,model_and_evidence_agree,None


In [20]:
# a follow-up interval written like Snellen: "6/52" fails the Snellen check, so the value row is in review;
# the blood pressure "183/71" is removed by the plausibility filter and never becomes a reading
show("BP 183/71 mmHg. Follow up in 6/52.")

BP 183/71 mmHg. Follow up in 6/52. 

letter status: needs_review


,va_value,lat,method,notation,label,start,end
0,6/52,Unk,Unk,snellen_m,VA-Unk-Unk,29,33


,va_value,field,final,model,confidence,state,reason,evidence
0,6/52,value,6/52,6/52,0.9680,review,value not plausible as snellen_m,None
1,6/52,lat,Unk,Unk,0.9996,unk,not_stated,None
2,6/52,method,Unk,Unk,0.9865,unk,not_stated,None
3,6/52,notation,snellen_m,snellen_m,0.9988,committed,model_and_evidence_agree,None


Filtering a batch to what needs a person:

In [21]:
batch = parse_many({
    'A': "Visual acuity with glasses: right eye 6/9, left eye 6/12.",
    'B': "VRE: 6/5, VLE: 6/5",
    'C': "Visual acuity: 73 on the right and 68 on the left.",
})
batch[batch['letter_status'] == 'needs_review']

,letter_id,letter_status,va_value,lat,method,notation,label,start,end
2,B,needs_review,6/5,Review,Unk,snellen_m,VA-Review-Unk,5,8
3,B,needs_review,6/5,Review,Unk,snellen_m,VA-Review-Unk,15,18


## Notes

- Thresholds: `thresholds.json` (calibrated on held-out validation letters for 99.5% committed precision). Passing `VABERTPipeline(thresholds={...})` overrides them; lower values commit more and review less.
- Command line: `vabert --text "..."`, `vabert --file letter.txt --json`, or `vabert --corpus letters.csv ...` (`python -m vabert ...` works without installing; see `README.md`).
- Results and limitations of this checkpoint are in `README.md`.